## Repository path configuration
All repository files are accessed through relative paths. Run the notebook from the repository root or from the `notebooks/` directory.


In [ ]:
from pathlib import Path
import os

# Resolve the repository root whether Jupyter starts in the repository root
# or directly inside the notebooks directory.
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)

DATA_DIR = ROOT / "data"
EXTERNAL_DATA_DIR = DATA_DIR / "external"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
OUTPUTS_DIR = ROOT / "outputs"
RESULTS_DIR = OUTPUTS_DIR / "results"
FIGURES_DIR = OUTPUTS_DIR / "figures"
EXTERNAL_MATERIALS_DIR = ROOT / "external_materials"
MODEL_WEIGHTS_DIR = EXTERNAL_MATERIALS_DIR / "model_weights"

for folder in [
    EXTERNAL_DATA_DIR,
    PROCESSED_DATA_DIR,
    PROCESSED_DATA_DIR / "logs",
    RESULTS_DIR,
    FIGURES_DIR,
    MODEL_WEIGHTS_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)

print("Repository root:", ROOT)


In [ ]:
# -----------------------------
# INSTALLS & IMPORTS
# -----------------------------
import os
os.environ["WANDB_API_KEY"] = os.getenv("WANDB_API_KEY", "")

!pip install wandb -qU  # Εγκατάσταση της πιο πρόσφατης έκδοσης της βιβλιοθήκης WandB.
import wandb            # Εισαγωγή της WandB.

# Αυτή η εντολή ενεργοποιεί τον χρήστη να συνδεθεί στο λογαριασμό του WandB.
# Θα χρειαστεί ένα API Key.
wandb.login(key=os.environ["WANDB_API_KEY"]) if os.getenv("WANDB_API_KEY") else None

# Google Drive mounting is intentionally not used in this repository.

!pip install -U transformers accelerate
!pip install bitsandbytes

import pandas as pd
import numpy as np
import torch
import random
import time
import nltk
import string
import matplotlib.pyplot as plt
import seaborn as sns

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer
from nltk.tag import pos_tag

from transformers import (
    AutoTokenizer,
    BartForSequenceClassification,
    Trainer,
    TrainingArguments,
    TrainerCallback,
    EarlyStoppingCallback
)

from torch.utils.data import Dataset
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

In [ ]:
# -----------------------------
# RANDOMNESS
# -----------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
# Κατέβασμα των απαραίτητων δεδομένων από το nltk
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')  # Για σωστό lemmatization
nltk.download('averaged_perceptron_tagger')  # POS tagging
nltk.download('punkt')  # Tokenizer
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')

In [ ]:
# -----------------------------
# TEXT CLEANING
# -----------------------------
def get_wordnet_pos(word):
    tag = pos_tag([word])[0][1][0].upper()
    tag_dict = {"J": wordnet.ADJ, "N": wordnet.NOUN,
                "V": wordnet.VERB, "R": wordnet.ADV}
    return tag_dict.get(tag, wordnet.NOUN)

def clean_text(text):
    stop_words = set(stopwords.words("english"))
    lemmatizer = WordNetLemmatizer()
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    words = word_tokenize(text)
    words = [w for w in words if w not in stop_words]
    words = [lemmatizer.lemmatize(w, get_wordnet_pos(w)) for w in words]
    return " ".join(words)

In [ ]:
# -----------------------------
# LOGGING CALLBACK
# -----------------------------
class LossLoggerCallback(TrainerCallback):
    def __init__(self):
        self.records = []

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None:
            return
        if "loss" in logs or "eval_loss" in logs:
            self.records.append({
                "epoch": logs.get("epoch"),
                "loss": logs.get("loss"),
                "eval_loss": logs.get("eval_loss")
            })

loss_logger = LossLoggerCallback()

In [ ]:
# -----------------------------
# LOAD DATA
# -----------------------------
df = pd.read_csv("data/external/sentences_dataset_45269.csv")

# Απενεργοποίηση καθαρισμού στα LLMs.
"""print("\nΠροτάσεις πριν τον καθαρισμό!\n")
print(df.head(30))

df["sentence"] = df["sentence"].apply(clean_text)

print("\nΠροτάσεις μετά τον καθαρισμό!\n")
print(df.head(30))"""

# Εμφάνιση δειγμάτων.
print("\nΠροτάσεις: \n")
print(df.head(30))

# Αφαίρεση διπλότυπων εγγραφών. Διατηρείται μόνο η πρώτη εμφάνιση της εν λόγω γραμμής.

# Υπολογισμός αριθμού διπλών εγγραφών.
num_duplicates = df['sentence'].duplicated().sum()
print("Πλήθος διπλών εγγραφών στη στήλη sentence:", num_duplicates)

# Αποθήκευση του αρχικού πλήθους γραμμών.
before = len(df)

# Αφαίρεση διπλότυπων (κρατάμε την πρώτη εμφάνιση).
df = df.drop_duplicates(subset='sentence', keep='first')

# Υπολογισμός του πόσες γραμμές διαγράφηκαν.
after = len(df)
deleted = before - after
print("Πλήθος εγγραφών που διαγράφηκαν:", deleted)

In [ ]:
# -----------------------------
# TRAIN/VAL/TEST SPLIT
# -----------------------------
train_df, temp_df = train_test_split(
    df,
    test_size=0.3,
    stratify=df["polarity"],
    random_state=SEED
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df["polarity"],
    random_state=SEED
)

# Εμφάνιση των διαστάσεων των σετ δεδομένων.
print("Διαστάσεις Σετ Εκπαίδευσης:", train_df.shape)
print("Διαστάσεις Σετ Ελέγχου:", test_df.shape)

print("Το πρώτο δείγμα του train είναι: ", train_df.head(1))

In [ ]:
# -----------------------------
# BART TOKENIZER
# -----------------------------
#MODEL_NAME = "facebook/bart-base"
MODEL_NAME = "facebook/bart-large"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

EPOCHS = int(input("Δώσε αριθμό εποχών εκπαίδευσης "))

# Δεκτές εποχές απο 3 έως 20.
while not EPOCHS >= 3 or not EPOCHS <=20:
    EPOCHS = int(input("Δώσε αριθμό εποχών εκπαίδευσης "))

In [ ]:
# -----------------------------
# DATASET CLASS
# -----------------------------
class BartClassificationDataset(Dataset):
    def __init__(self, sentences, labels):
        self.encodings = tokenizer(
            sentences,
            truncation=True,
            padding=True,
            max_length=256
        )
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx])
                for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [ ]:
# Create datasets
train_dataset = BartClassificationDataset(
    train_df["sentence"].tolist(),
    train_df["polarity"].tolist()
)

val_dataset = BartClassificationDataset(
    val_df["sentence"].tolist(),
    val_df["polarity"].tolist()
)

test_dataset = BartClassificationDataset(
    test_df["sentence"].tolist(),
    test_df["polarity"].tolist()
)

In [ ]:
# -----------------------------
# LOAD BART CLASSIFICATION MODEL
# -----------------------------
model = BartForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    #dtype=torch.float16 if torch.cuda.is_available() else None
)


In [ ]:
# -----------------------------
# TRAINING ARGS
# -----------------------------
"""training_args = TrainingArguments(
    output_dir="./results_bart",
    run_name="bart_sentiment_classifier",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=4,
    warmup_ratio=0.1,
    learning_rate=1e-5,
    weight_decay=0.01,
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=True,
    report_to="none"
)"""


training_args = TrainingArguments(
    output_dir='./results',
    run_name="Sentiment_Analysis_BART",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    logging_dir='./logs',
    warmup_ratio=0.1,
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    gradient_accumulation_steps=4,
    report_to="none",
    save_total_limit=2,
    #max_grad_norm=1.0,
    learning_rate=1e-5,
)

In [ ]:
# -----------------------------
# TRAINER
# -----------------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    callbacks=[loss_logger, EarlyStoppingCallback(early_stopping_patience=4)]
)

In [ ]:
# -----------------------------
# TRAIN
# -----------------------------

start_time = time.time()
print("\nΞεκινά η εκπαίδευση...\n")

trainer.train()

end_time = time.time()
minutes = int((end_time - start_time) // 60)
seconds = int((end_time - start_time) % 60)
print(f"\nΣυνολικός χρόνος: {minutes} λεπτά {seconds} δευτερόλεπτα\n")

In [ ]:
# ============================================
#  LOGS TO CSV 
# ============================================

records = loss_logger.records
df_logs = pd.DataFrame(records)

train_df = df_logs[df_logs["loss"].notna()][["epoch", "loss"]].rename(columns={"loss": "train_loss"})
eval_df = df_logs[df_logs["eval_loss"].notna()][["epoch", "eval_loss"]].rename(columns={"eval_loss": "val_loss"})

df_loss = pd.merge(train_df, eval_df, on="epoch", how="left")
df_loss = df_loss.sort_values("epoch").reset_index(drop=True)

NAME = MODEL_NAME.replace("/", "_")
csv_path = f"data/processed/logs/loss_{NAME}.csv"
df_loss.to_csv(csv_path, index=False)
print("Saved loss CSV:", csv_path)

In [ ]:
# ============================================
#  SAVE MODEL 
# ============================================

model_path = f"external_materials/model_weights/{NAME}/saved_model"
tokenizer_path = f"external_materials/model_weights/{NAME}/saved_tokenizer"

model.save_pretrained(model_path)
tokenizer.save_pretrained(tokenizer_path)

print(f"Το μοντέλο '{MODEL_NAME}' και ο tokenizer αποθηκεύτηκαν επιτυχώς!")

In [ ]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

In [ ]:
# -----------------------------
# EVALUATION
# -----------------------------
from torch.cuda.amp import autocast

from torch.utils.data import DataLoader, Dataset

class TextDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return self.texts[idx], self.labels[idx]

dataset = TextDataset(test_df["sentence"].tolist(), test_df["polarity"].tolist())
dataloader = DataLoader(dataset, batch_size=16, shuffle=False)


model.eval()
preds = []
true = []

with torch.no_grad():
    for batch_texts, batch_labels in dataloader:

        enc = tokenizer(
            batch_texts,
            truncation=True,
            padding=True,
            max_length=256,
            return_tensors="pt"
        ).to(model.device)

        logits = model(**enc).logits
        batch_preds = torch.argmax(logits, dim=1).cpu().tolist()

        preds.extend(batch_preds)
        true.extend(batch_labels)


print("\nClassification Report:\n")
report = classification_report(true, preds, target_names=["Negative", "Neutral", "Positive"])
print(report)

with open(f"outputs/results/classification_reports/{NAME}_classification_report.txt", "w", encoding="utf-8") as f:
        f.write(report)

plt.rcParams.update({'font.size': 11})
plt.figure(figsize=(6, 4))

cm = confusion_matrix(true, preds)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Negative","Neutral","Positive"],
            yticklabels=["Negative","Neutral","Positive"])

plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()

plt.savefig(
        f'outputs/figures/fig/{NAME}_heatmap_SA_citation.pdf',
        format='pdf',
        bbox_inches='tight',
        dpi=300
    )

#plt.title("Confusion Matrix")
plt.show()

In [ ]:
#############################